# Introduction

This notebook demonstrates how to write academic papers using Jupyter Notebooks with [Quarto](https://quarto.org/) and [MakeTables](https://py-econometrics.github.io/maketables/). When you call `table.make()` without a `type` argument, MakeTables automatically shows a formatted HTML table in the notebook and renders a LaTeX table when Quarto compiles the document to PDF.

In [ ]:
import numpy as np
import pandas as pd
import pyfixest as pf
import statsmodels.formula.api as smf
import maketables as mt

# Load the sample dataset
df = pd.read_csv("../../data/salaries.csv")

# Set variable labels used throughout the paper
mt.MTable.DEFAULT_LABELS = {
    "logwage": "Log Wage",
    "wage": "Wage",
    "age": "Age",
    "female": "Female",
    "tenure": "Tenure",
    "occupation": "Occupation",
    "worker_type": "Worker Type",
    "education": "Education",
    "promoted": "Promoted",
}

# Create a gender variable from the female dummy
df["gender"] = df["female"].map({0: "Male", 1: "Female"})

# Data Description

We use a sample of 1,800 workers with information on wages, demographics, and employment characteristics. Table 1 presents summary statistics by worker type.

In [ ]:
tab1 = mt.DTable(
    df,
    vars=["wage", "age", "tenure"],
    bycol=["worker_type"],
    byrow="gender",
    stats=["count", "mean", "std"],
    caption="Summary Statistics by Worker Type and Gender",
    tab_label="tab:descriptives",
    format_spec={"mean": ",.2f", "std": ".2f"},
)
tab1.make()

# Wage Regressions

Table 2 presents OLS estimates of the log wage equation. We progressively add controls for age, gender, and their interaction.

In [ ]:
tab2 = mt.ETable(
    pf.feols("logwage ~ age + female + sw0(age:female)", data=df),
    caption="Log Wage Regressions",
    tab_label="tab:regressions",
)
tab2.make()

# Promotion Analysis

Table 3 compares OLS and Probit estimates for the probability of promotion.

In [ ]:
est1 = smf.ols("promoted ~ tenure + female + worker_type", data=df).fit()
est2 = smf.probit("promoted ~ tenure + female + worker_type", data=df).fit(disp=0)

tab3 = mt.ETable(
    [est1, est2],
    keep=["tenure", "female", "worker_type"],
    model_stats=["N", "r2", "pseudo_r2"],
    model_heads=["OLS", "Probit"],
    caption="Predicting Promotions",
    tab_label="tab:promotions",
)
tab3.make()